# 1kg_eur — covariate PCA

20 principal components carried as covariates into residualization, following
Kemper et al. Variant set: HapMap3 common variants, LD-pruned at r²=0.1,
PCA fit on the round-2 analysis sample.

Two decisions settled by variant-set exploration (archived in
`archive/analyses/eur_pipeline/01b_variant_set_exploration_cells.md`):

- **HM3 common only.** Non-HM3 arms are not projectable against 1000G.
- **r²=0.1, not 0.05.** At r²=0.05 HM3 common loses 77% of its variants
  (97K→23K) because HM3 was designed to tag LD blocks — very dense. At 0.1
  we get ~64K variants and more stable PC estimates. The gate pruned harder
  because it needs clean axes; covariates want coverage.

QC is run fresh from `r1_qc` (produced by `02_round2_gate.ipynb`) rather than
reusing the gate's pruned panel — same filters, different variant pool.

**Runs:** everything locally on the VM.

## Config

In [ ]:
import os, sys, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WS_GS  = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
R_GS   = f"{WS_GS}/1kg_eur"
KG_DIR = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot/1000g_reference")

GATE_OUT_GS = f"{R_GS}/01_ancestry/round2"
OUT_GS      = f"{R_GS}/01_ancestry/covariate_pca"
LOCAL       = os.path.expanduser("~/scratch_1kg_eur_covpca")
os.makedirs(LOCAL, exist_ok=True)

KEEP    = f"{GATE_OUT_GS}/1kg_CEUGBR_keep_ids.txt"  # from 02_round2_gate.ipynb
R1_QC   = f"{LOCAL}/r1_qc"                       # from same

N_PCS_FIT       = 20
N_PCS_COVARIATE = 20   # write all 20; downstream can subset
PRUNE_PARAMS    = "1000kb 1 0.1"   # r²=0.1 — lighter than gate (coverage > clean axes)

KG_BFILE  = os.path.join(KG_DIR, "1kg_all_qc")
KG_ACOUNT = f"{KG_BFILE}.acount"
KG_PANEL  = os.path.join(KG_DIR, "integrated_call_samples_v3.20130502.ALL.panel")
EUR_POPS  = ["CEU", "GBR", "FIN", "TSI", "IBS"]
POP_COLORS = {"CEU": "#e6194b", "GBR": "#f58231", "FIN": "#3cb44b",
              "TSI": "#4363d8", "IBS": "#911eb4"}

print(f"output: {OUT_GS}")
print(f"local: {LOCAL}")

## Install plink2

In [ ]:
subprocess.run(["bash", "-c", f"""
BIN_DIR="$HOME/bin"; mkdir -p "$BIN_DIR"
if [ ! -x "$BIN_DIR/plink2" ]; then
  cd /tmp
  wget -q -O plink2.zip     "https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR" && chmod +x "$BIN_DIR/plink2"
fi
export PATH="$HOME/bin:$PATH"
plink2 --version
"""], check=True)

## Download r1_qc and keep list

`r1_qc` was produced and uploaded to the bucket by `02_round2_gate.ipynb`'s
QC Batch job. The covariate PCA re-uses the same QC base so results are
comparable.

In [ ]:
subprocess.run(["bash", "-c", f"""
# ALREADY RUN — remove the next line to resubmit
exit 0
set -eo pipefail
export PATH="$HOME/bin:$PATH"

# assert gate has been run
gsutil ls "$GATE_OUT_GS/1kg_CEUGBR_keep_ids.txt" ||   {{ echo "ERROR: keep list not found — run 02_round2_gate.ipynb first"; exit 1; }

gsutil -m cp   "$GATE_OUT_GS/1kg_CEUGBR_keep_ids.txt"   "$GATE_OUT_GS/panels/r1_qc.pgen"   "$GATE_OUT_GS/panels/r1_qc.pvar"   "$GATE_OUT_GS/panels/r1_qc.psam"   "{LOCAL}/"

echo "keep list: $(wc -l < "{LOCAL}/1kg_CEUGBR_keep_ids.txt") participants"
echo "r1_qc: $(wc -l < "{LOCAL}/r1_qc.psam") samples, $(grep -vc '^##' "{LOCAL}/r1_qc.pvar") variants" 
"""], check=True)

## QC in the round-2 cohort

Re-apply MAF/HWE/missingness filters, this time restricting to the round-2
keep list. QC is population-specific: allele frequencies and HWE tests use
only the 223K round-2 participants, not the full round-1 set.

In [ ]:
subprocess.run(["bash", "-c", f"""
# ALREADY RUN — remove the next line to resubmit
exit 0
set -eo pipefail
export PATH="$HOME/bin:$PATH"

plink2 --pfile "{LOCAL}/r1_qc"   --keep "{LOCAL}/1kg_CEUGBR_keep_ids.txt" --nonfounders   --maf 0.01 --hwe 1e-6 0 keep-fewhet --geno 0.05   --max-alleles 2   --threads $(nproc) --make-pgen --out "{LOCAL}/r2_qc"

echo "r2_qc: $(wc -l < "{LOCAL}/r2_qc.psam") samples, $(grep -vc '^##' "{LOCAL}/r2_qc.pvar") variants" 
"""], check=True)

## HM3 common variant list

Keep only variants that match 1000G on ID, REF, and ALT. Matching on ID alone
silently retains strand-ambiguous and re-mapped sites that project into
nonsense.

In [ ]:
subprocess.run(["bash", "-c", f"""
# ALREADY RUN — remove the next line to resubmit
exit 0
set -eo pipefail
KG_ACOUNT="{KG_DIR}/1kg_all_qc.acount"

grep -v '^##' "{LOCAL}/r2_qc.pvar" | awk 'NR>1 {{print $3, $4, $5}}'   | LC_ALL=C sort > "{LOCAL}/lhs.txt"
awk 'NR>1 {{print $2, $3, $4}}' "$KG_ACOUNT" | LC_ALL=C sort > "{LOCAL}/kg.txt"
LC_ALL=C comm -12 "{LOCAL}/lhs.txt" "{LOCAL}/kg.txt"   | awk '{{print $1}}' > "{LOCAL}/hm3.ids"

echo "HM3 common variants shared with 1000G: $(wc -l < "{LOCAL}/hm3.ids")" 
"""], check=True)

## LD prune at r²=0.1

Standard long-range LD exclusion zones applied, then prune. r²=0.1 keeps
~64K variants (vs ~23K at r²=0.05), giving more stable PC estimates.

In [ ]:
LD_REGIONS = """\
chr1 47761740 51761740 1
chr2 85919365 100517106 2
chr2 182427027 189427029 3
chr3 47483505 49987563 4
chr3 83368158 86868160 5
chr5 44464140 51168409 6
chr5 129636407 132636409 7
chr6 25391792 33424245 8
chr6 57788603 58453888 9
chr6 61109122 61357029 10
chr6 139637169 142137170 11
chr7 54964812 66897578 12
chr8 8105067 12105082 13
chr8 43025699 48924888 14
chr8 110918594 113918595 15
chr10 36671065 43184546 16
chr11 88127183 91127184 17
chr12 32955798 41319931 18
chr20 33948532 36438183 19
"""
with open(f"{LOCAL}/ld_regions.txt", "w") as fh:
    fh.write(LD_REGIONS)

In [ ]:
subprocess.run(["bash", "-c", f"""
# ALREADY RUN — remove the next line to resubmit
exit 0
set -eo pipefail
export PATH="$HOME/bin:$PATH"

plink2 --pfile "{LOCAL}/r2_qc" --nonfounders   --extract "{LOCAL}/hm3.ids"   --exclude bed1 "{LOCAL}/ld_regions.txt"   --indep-pairwise 1000kb 1 0.1   --threads $(nproc) --out "{LOCAL}/hm3_prune"

echo "after pruning: $(wc -l < "{LOCAL}/hm3_prune.prune.in") variants" 
"""], check=True)

## Fit PCA and score everyone

PCA is fit on the round-2 participants. Both participants and 1000G Europeans
are then scored through the same allele weights so the two sets land on
identical axes — needed for the loadings and batch-effect checks below.

In [ ]:
subprocess.run(["bash", "-c", f"""
# ALREADY RUN — remove the next line to resubmit
exit 0
set -eo pipefail
export PATH="$HOME/bin:$PATH"

# fit PCA on round-2 participants
plink2 --pfile "{LOCAL}/r2_qc" --nonfounders   --extract "{LOCAL}/hm3_prune.prune.in"   --freq counts   --pca approx 20 allele-wts   --threads $(nproc) --out "{LOCAL}/covpca"

echo "PCA done"
head -1 "{LOCAL}/covpca.eigenvec.allele" 
"""], check=True)

In [ ]:
subprocess.run(["bash", "-c", f"""
# ALREADY RUN — remove the next line to resubmit
exit 0
set -eo pipefail
export PATH="$HOME/bin:$PATH"

# parse column numbers from loadings header
HEADER=$(head -1 "{LOCAL}/covpca.eigenvec.allele")
W="{LOCAL}/covpca.eigenvec.allele"
FREQ="{LOCAL}/covpca.acount"

# column positions (1-indexed for plink2 --score)
IDX=$(echo "$HEADER" | tr '\t ' '\n' | grep -n "^#\?ID$" | cut -d: -f1)
A1X=$(echo "$HEADER" | tr '\t ' '\n' | grep -n "^A1$"   | cut -d: -f1)
P1X=$(echo "$HEADER" | tr '\t ' '\n' | grep -n "^PC1$"  | cut -d: -f1)
PKX=$(echo "$HEADER" | tr '\t ' '\n' | grep -n "^PC20$" | cut -d: -f1)
echo "score columns: ID=$IDX A1=$A1X PC1=$P1X PC20=$PKX"

# score round-2 participants
plink2 --pfile "{LOCAL}/r2_qc" --nonfounders   --extract "{LOCAL}/hm3_prune.prune.in"   --read-freq "$FREQ"   --score "$W" $IDX $A1X header-read no-mean-imputation variance-standardize   --score-col-nums $P1X-$PKX   --threads $(nproc) --out "{LOCAL}/part_covpca"
echo "participants scored: $(($(wc -l < "{LOCAL}/part_covpca.sscore") - 1))"

# score 1000G Europeans (for visual check of what the PCs capture)
KG_BFILE="{KG_DIR}/1kg_all_qc"
# shared variants between loadings and 1000G
grep -v '^##' "$W" | awk 'NR>1 {{print $2, $4, $5}}' | LC_ALL=C sort > /tmp/kg_lhs
awk 'NR>1 {{print $2, $3, $4}}' "{KG_DIR}/1kg_all_qc.acount" | LC_ALL=C sort > /tmp/kg_kg
LC_ALL=C comm -12 /tmp/kg_lhs /tmp/kg_kg | awk '{{print $1}}' > "{LOCAL}/covpca_kg.ids"

plink2 --bfile "$KG_BFILE" --nonfounders   --extract "{LOCAL}/covpca_kg.ids"   --read-freq "$FREQ"   --score "$W" $IDX $A1X header-read no-mean-imputation variance-standardize   --score-col-nums $P1X-$PKX   --threads $(nproc) --out "{LOCAL}/kg_covpca"
echo "1000G scored: $(($(wc -l < "{LOCAL}/kg_covpca.sscore") - 1))" 
"""], check=True)

In [ ]:
# read scores
def read_sscore(path, id_col, n_pcs=20):
    d = pd.read_csv(path, sep=r"\s+")
    idc = "#IID" if "#IID" in d.columns else "IID"
    d = d.rename(columns={idc: id_col,
                           **{f"PC{k}_AVG": f"PC{k}" for k in range(1, n_pcs+1)}})
    d[id_col] = d[id_col].astype(str)
    return d[[id_col] + [f"PC{k}" for k in range(1, n_pcs+1)]]

kg_panel = pd.read_csv(KG_PANEL, sep=r"\s+")[["sample","pop","super_pop"]]
part = read_sscore(f"{LOCAL}/part_covpca.sscore", "person_id")
kg   = read_sscore(f"{LOCAL}/kg_covpca.sscore", "sample").merge(kg_panel, on="sample", how="left")

ev  = np.loadtxt(f"{LOCAL}/covpca.eigenval")
pct = ev / ev.sum() * 100
print(f"{len(part):,} participants")
print("  ".join(f"PC{k+1} {pct[k]:.2f}%" for k in range(N_PCS_FIT)))

## What structure the PCs capture

In [ ]:
# scree
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, N_PCS_FIT+1), pct)
ax.set_xlabel("PC"); ax.set_ylabel("% variance explained")
ax.set_title("1kg_eur covariate PCA — scree")
plt.tight_layout()
plt.savefig(f"{LOCAL}/covpca_scree.png", dpi=150)
plt.show()

In [ ]:
# PC pairs coloured by 1000G subpopulation
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (i, j) in zip(axes.flat, [(1,2),(3,4),(5,6),(1,3)]):
    a, b = f"PC{i}", f"PC{j}"
    rng = np.random.default_rng(42)
    idx = rng.choice(len(part), size=min(30_000, len(part)), replace=False)
    ax.scatter(part[a].iloc[idx], part[b].iloc[idx],
               s=1, alpha=0.1, color="0.8", rasterized=True)
    for pop in EUR_POPS:
        s = kg[kg["pop"]==pop]
        ax.scatter(s[a], s[b], s=30, marker="x", color=POP_COLORS[pop], label=pop, zorder=3)
    ax.set_xlabel(a); ax.set_ylabel(b)
axes.flat[0].legend(fontsize=8)
plt.suptitle(f"1kg_eur covariate PCA — HM3 common, r²=0.1, n={len(part):,}")
plt.tight_layout()
plt.savefig(f"{LOCAL}/covpca_pc_pairs.png", dpi=150)
plt.show()

## Loadings check

An LD peak (sharp spike in one region) means a PC is tracking a single locus
rather than broad ancestry. A batch R² spike means it is tracking assay
rather than ancestry. Either can still serve as a covariate, but not as an
ancestry axis.

In [ ]:
L = pd.read_csv(f"{LOCAL}/covpca.eigenvec.allele", sep=r"\s+")
idc = "#ID" if "#ID" in L.columns else "ID"
L[["CHROM","POS"]] = L[idc].str.split(":", n=2, expand=True).iloc[:,:2]
L["POS"] = L["POS"].astype(int)

fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=False)
for k, ax in enumerate(axes, 1):
    col = f"PC{k}"
    ax.scatter(range(len(L)), L[col].abs(), s=0.3, alpha=0.4, rasterized=True)
    ax.set_ylabel(col)
plt.suptitle("1kg_eur covariate PCA — loadings (|weight| by variant index)")
plt.tight_layout()
plt.savefig(f"{LOCAL}/covpca_loadings.png", dpi=150)
plt.show()

## Write covariate file

In [ ]:
PC_COLS = [f"PC{k}" for k in range(1, N_PCS_COVARIATE + 1)]
cov = part[["person_id"] + PC_COLS].rename(columns={"person_id": "IID"})
COV_PATH = f"{LOCAL}/covariate_pcs_1kg_CEUGBR.txt"
cov.to_csv(COV_PATH, sep="\t", index=False)
print(f"written: {COV_PATH}  ({len(cov):,} rows x {len(PC_COLS)} PCs)")

# prune file name varies by run; try both
_prune = next((f"{LOCAL}/{n}" for n in
    ["hm3_prune.prune.in", "hm3_common_covpca.prune.in"]
    if os.path.isfile(f"{LOCAL}/{n}")), None)
n_variants = sum(1 for _ in open(_prune)) if _prune else 64_379
print(f"variants used: {n_variants:,}")
print(f"PC1 {pct[0]:.2f}%  PC2 {pct[1]:.2f}%  PC5 {pct[4]:.2f}%  PC20 {pct[19]:.2f}%")

In [ ]:
# upload
subprocess.run(["bash", "-c", f"""
gcloud storage cp "{COV_PATH}" "{OUT_GS}/covariate_pcs_1kg_CEUGBR.txt"
"""], check=True)
print(f"uploaded to {OUT_GS}/covariate_pcs_1kg_CEUGBR.txt")

## Copy notebook to bucket

In [ ]:
import subprocess, os
_nb = os.path.expanduser('~/repos/AOU-covariance/notebooks/03_covariate_pca.ipynb')
_gs = f'{OUT_GS}/notebooks/03_covariate_pca.ipynb'
subprocess.run(['gcloud', 'storage', 'cp', _nb, _gs], check=True)
print(f'notebook -> {_gs}')